# Git Commit-Message Generator -- notebook demo

This notebook is a runnable demo of the **Git Commit-Message Generator** project from
the course: [`docs/projects/commit-message-agent/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/commit-message-agent),
companion to the fuller local CLI at [`examples/commit-message-agent/commit_helper.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/commit-message-agent/commit_helper.py).

It captures a real `git diff`, hands it to a free-tier LLM with a commit-message-drafting
system prompt, and prints back a Conventional-Commits-style message -- a short imperative
summary line, plus an optional body explaining why the change was made.


## A note on running this in a notebook

The real version of this tool (`examples/commit-message-agent/commit_helper.py`) drafts a
message for **your own staged changes** in a real local git repository -- `git diff --staged`
-- and then, only after you type `y` at a confirmation prompt, runs `git commit -m "..."` for
you. That interactive accept/edit/commit loop is the whole point of the tool, and it needs a
real repo you're actually working in.

Colab and Kaggle don't give you that -- there's no local repo of your own with staged changes
here, just this notebook's ephemeral, ready-made environment. So **this demo adapts the
tool**: instead of drafting a message for your own staged work, it shallow-clones this
course's own repository right here in the notebook and drafts a message for one real, small,
historical commit from it with `git show`. That's a legitimate demo of the drafting logic (the
`subprocess` diff capture, the system prompt, the LLM call) -- it does **not** demo the
interactive accept/edit/commit loop, since committing only makes sense against a repo you're
really working in. Come back to `uv run python commit_helper.py` locally, or a
[Codespace](https://codespaces.new/abderrahim-lectures/python-data-analysis-course), for that
part.


In [ ]:
!pip install -q openai

## Get a real diff to draft a message for

Shallow-clone this course's own repository and pick one real, small, illustrative commit from
its actual history: a CSS bug fix (`be914ee`, "Fix hamburger menu not opening on desktop
browsers") that touches one file with a focused, explainable change -- a good size and shape
for a demo draft.


In [ ]:
!git clone --depth 50 https://github.com/abderrahim-lectures/python-data-analysis-course.git /tmp/repo

In [ ]:
import subprocess

DEMO_COMMIT = "be914ee44ce104124f38b775ff200072e91095bd"  # "Fix hamburger menu not opening on desktop browsers"


def get_diff_for_commit(commit: str, repo_dir: str = "/tmp/repo") -> str:
    """The diff introduced by one specific past commit, vs. its parent.

    Same idea as `get_diff_for_commit` in commit_helper.py, just pointed at
    the freshly-cloned repo above instead of the current directory.
    """
    result = subprocess.run(
        ["git", "-C", repo_dir, "show", commit],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"git show {commit} failed:\n{result.stderr}")
    return result.stdout


demo_diff = get_diff_for_commit(DEMO_COMMIT)
print(demo_diff[:1500])

## The commit-message system prompt

This is the exact `SYSTEM_PROMPT` from `commit_helper.py` -- it's what turns a general-purpose
chat model into a focused commit-message drafter: Conventional Commits style, imperative mood,
a short summary line, and an optional body only when it adds real information.


In [ ]:
SYSTEM_PROMPT = """\
You are an experienced software engineer writing a git commit message for a
staged diff. You will be given a unified git diff. Base the message ONLY on
what the diff actually changes -- do not invent context you can't see, and
do not guess at a ticket number or issue reference that isn't in the diff.

Write the message in the Conventional Commits style:

    <type>(<optional scope>): <short summary, imperative mood, no period>

    <optional body: a few lines explaining WHY the change was made, not
    just restating what the diff shows -- wrap around 72 characters>

Valid types: feat, fix, docs, style, refactor, perf, test, build, ci, chore.
Pick the type that best matches the *dominant* change -- if a diff touches
both a fix and its test, "fix" usually still wins over "test".

Rules:
- The summary line must stay under 72 characters and use the imperative
  mood ("add", not "added" or "adds").
- Only include a body if it adds real information beyond the summary --
  for a small, self-explanatory diff, the summary line alone is enough.
- Never wrap the whole message in a fenced code block or add commentary
  before/after it -- output ONLY the commit message text itself, nothing
  else, so it can be used directly as a commit message.
"""

## Get a free-tier API key

This demo defaults to **GitHub Models** -- free, no separate signup, just a personal access
token with the `models: read` scope from [github.com/settings/tokens](https://github.com/settings/tokens).
Any of the other five providers wired up in `commit_helper.py` (Gemini, Groq, Mistral,
Cerebras, OpenRouter) work too -- see that file's `PROVIDERS` dict for their base URLs and env
var names, and adjust `LLM_PROVIDER` below.

The key is entered with `getpass` so it never gets typed into a visible cell or saved into
this notebook's output -- never hardcode a real API key here.


In [ ]:
import os
from getpass import getpass

LLM_PROVIDER = "github"  # change to gemini / groq / mistral / cerebras / openrouter if you prefer
os.environ["GITHUB_TOKEN"] = getpass("Enter your free-tier GitHub Models token (GITHUB_TOKEN): ")

## The drafting logic itself

This mirrors `truncate_diff`, `PROVIDERS`, and `draft_commit_message` from `commit_helper.py`
directly -- the same truncation cap, the same free-tier providers (all exposed through the
`openai` client, just pointed at each provider's own OpenAI-compatible endpoint), and the same
call shape. Note that `draft_commit_message` only ever returns a string -- exactly like in the
real script, nothing here calls `git commit`.


In [ ]:
from openai import OpenAI

MAX_DIFF_CHARS = 12_000


def truncate_diff(diff: str, max_chars: int = MAX_DIFF_CHARS) -> str:
    """Cuts an oversized diff down to a size that fits a free-tier context window."""
    if len(diff) <= max_chars:
        return diff
    return diff[:max_chars] + f"\n\n... [diff truncated -- {len(diff) - max_chars} more characters not shown] ..."


def _build_github_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.github.ai/inference")


def _build_gemini_client() -> OpenAI:
    return OpenAI(
        api_key=os.environ["GOOGLE_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )


def _build_groq_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")


def _build_mistral_client() -> OpenAI:
    return OpenAI(api_key=os.environ["MISTRAL_API_KEY"], base_url="https://api.mistral.ai/v1")


def _build_cerebras_client() -> OpenAI:
    return OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")


def _build_openrouter_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")


PROVIDERS = {
    "github": (_build_github_client, "gpt-4o-mini"),
    "gemini": (_build_gemini_client, "gemini-3.5-flash"),
    "groq": (_build_groq_client, "llama-3.3-70b-versatile"),
    "mistral": (_build_mistral_client, "mistral-small-latest"),
    "cerebras": (_build_cerebras_client, "llama-3.3-70b"),
    "openrouter": (_build_openrouter_client, "meta-llama/llama-3.3-70b-instruct:free"),
}


def draft_commit_message(diff: str, provider: str | None = None) -> str:
    """Sends a diff to a free-tier LLM with the commit-message system prompt and returns a draft string."""
    if not diff.strip():
        return ""

    provider = provider or os.environ.get("LLM_PROVIDER", "github")
    build_client, model = PROVIDERS[provider]
    client = build_client()

    diff = truncate_diff(diff)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Write a commit message for this staged diff:\n\n```diff\n{diff}\n```"},
        ],
    )
    return response.choices[0].message.content.strip()

## Draft the message

Drafting for the demo commit cloned above -- a real historical commit from this course's own
repository, not the student's own staged work (see the note near the top of this notebook).
Nothing here calls `git commit` -- this notebook only demos the drafting step, not the
interactive accept/edit/commit loop from `commit_helper.py`.


In [ ]:
print(f"Drafting a commit message from {len(demo_diff)} characters of diff for commit {DEMO_COMMIT[:7]}...\n")
draft = draft_commit_message(demo_diff, provider=LLM_PROVIDER)
print(draft)